In [0]:
dbutils.widgets.text("catalog_name", "allianz_coe")
dbutils.widgets.text("control_schema_name", "audit_control")
dbutils.widgets.text("watermark_table_name", "application_watermark")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
control_schema_name = dbutils.widgets.get("control_schema_name")
watermark_table_name = dbutils.widgets.get("watermark_table_name")

In [0]:
full_wm_table_name = f"{catalog_name}.{control_schema_name}.{watermark_table_name}"

/Allianz/Allianz_COE/ETL_Ingestion/notebooks/dim/dim_loader_scd2


In [0]:
%run /Workspace/Allianz/Allianz_COE/ETL_Ingestion/utils/utils_nb

In [0]:
created_by="ETL_SYSTEM"
"""etl_component_name = dbutils.notebook.entry_point.getDbutils() \
.notebook().getContext().notebookPath().get()
print(etl_component_name)"""

def _as_list(x):
        return x if isinstance(x, (list, tuple)) else [x]

def _join_on_keys(left_alias, right_alias, key_cols):
        return " AND ".join([f"{left_alias}.{c} <=> {right_alias}.{c}" for c in key_cols])

def _key_select(key_cols):
        return ", ".join(key_cols)



/Allianz/Allianz_COE/ETL_Ingestion/notebooks/dim/dim_loader_scd2


In [0]:
# ============================================================
# SCD TYPE 2 - Dimension Loader 
# ============================================================
def add_record_version( target_table: str,
                        src_view: str,
                        out_view: str,
                        business_key_cols,
                        effective_from_col: str,
                        batch_run_id,
                        task_run_id,
                        record_version_col: str = "record_version"
                        ):
        """
        Compute record_version for each incoming SCD2 version:
        record_version = COALESCE(max_version_in_target, 0) + row_number() over (partition by BK order by effective_from)
        This satisfies:
        - new BK => max_version null => 0 + 1 = 1
        - changed BK => max_version + 1
        - multiple incoming versions in same batch => + row_number increment
        """
        bk_cols = _as_list(business_key_cols)
        bk_sel = _key_select(bk_cols)
        join_keys = _join_on_keys("i", "m", bk_cols)

        out_view_query = f"""
        CREATE OR REPLACE TEMP VIEW {out_view} AS
        WITH maxv AS (
            SELECT {bk_sel}, MAX({record_version_col}) AS max_ver
            FROM {target_table}
            GROUP BY {bk_sel}
        ),
        incoming AS (
            SELECT
              s.*,
              ROW_NUMBER() OVER (
                PARTITION BY {", ".join([f"s.{c}" for c in bk_cols])}
                ORDER BY s.{effective_from_col}, s.attr_hash
              ) AS rn
            FROM {src_view} s
        )
        SELECT
          i.*,
          COALESCE(m.max_ver, 0) + i.rn AS {record_version_col}
        FROM incoming i
        LEFT JOIN maxv m
          ON {join_keys}
        """

        log_message = f"creating out_view_query using ----- {out_view_query} "
        task_log(task_run_id=task_run_id,task_name=target_table,etl_component_name=etl_component_name,message=log_message)
        # logging.info("------ out_view_query ------")
        # logging.info(out_view_query)
        spark.sql(out_view_query)

# def __init__(self, spark,batch_run_id, task_run_id,created_by="ETL_SYSTEM"):
#         spark = spark
#         created_by = created_by
#         batch_run_id = batch_run_id
#         task_run_id = task_run_id

def get_watermark(target_table: str, watermark_col: str,batch_run_id,task_run_id) -> str:
    # ### CHANGED: Use TIMESTAMP fallback when watermark_col is timestamp
    # wm = spark.sql(f"""
    #     SELECT COALESCE(MAX({watermark_col}), TIMESTAMP('1900-01-01 00:00:00')) AS wm
    #     FROM {target_table}
    # """).collect()[0]["wm"]

    wm = spark.sql(f"""
        SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS wm
        FROM {full_wm_table_name} WHERE table_name = '{target_table}'
    """).collect()[0]["wm"]

    log_message = f"Watermark used inside get_watermark() : {wm}"
    task_log(task_run_id=task_run_id,task_name=target_table,etl_component_name=etl_component_name,message=log_message)
    return str(wm)

def build_stage_view( stage_sql: str, stage_view: str,batch_run_id,task_run_id,target):
    build_stage_view_query = f"CREATE OR REPLACE TEMP VIEW {stage_view} AS {stage_sql}"

    log_message = f"creating build_stage_view using ----- {build_stage_view_query} "
    task_log(task_run_id=task_run_id,task_name=target,etl_component_name=etl_component_name,message=log_message)
    # logging.info("------ build_stage_view_query ------")
    # logging.info(build_stage_view_query)
    spark.sql(build_stage_view_query)

def compute_attr_hash( stage_view: str, hash_cols: list, out_view: str,batch_run_id,task_run_id,target):
    # Create a view with attr_hash
    expr = "sha2(concat_ws('||'," + ",".join([f"coalesce(cast({c} as string),'')" for c in hash_cols]) + "),256)"
    compute_attr_hash_query = f"""
        CREATE OR REPLACE TEMP VIEW {out_view} AS
        SELECT
        *,
        {expr} AS attr_hash
        FROM {stage_view}
    """

    log_message = f"creating compute_attr_hash_query using ----- {compute_attr_hash_query} "
    task_log(task_run_id=task_run_id,task_name=target,etl_component_name=etl_component_name,message=log_message)
    # logging.info("------ compute_attr_hash_query ------")
    # logging.info(compute_attr_hash_query)
    spark.sql(compute_attr_hash_query)

# ----------------------------------------------------
# ### CHANGED: expire uses composite keys + correct eff_to
# ----------------------------------------------------

def expire_changed_current(
                              target_table: str,
                              src_view: str,
                              business_key_cols,
                              target_current_flag_col: str,
                              target_attr_hash_col: str,
                              target_eff_to_col: str,
                              src_eff_from_col: str,
                              audit_cols: dict,
                              batch_run_id,
                              task_run_id):
        # Expire if attr_hash differs
        # effective_to_ts = src.effective_from_ts - 1 second

        bk_cols = _as_list(business_key_cols)
        bk_sel = _key_select(bk_cols)
        on_keys = _join_on_keys("tgt", "src", bk_cols)

        upd_by = audit_cols.get("last_updated_by", "last_updated_by")
        upd_ts = audit_cols.get("last_updated_ts", "last_updated_ts")

        expire_changed_current_query = f"""
          MERGE INTO {target_table} AS tgt
          USING (
            SELECT {bk_sel},
                   {src_eff_from_col} AS new_eff_from,
                   attr_hash AS new_hash
            FROM {src_view}
          ) AS src
          ON {on_keys}
         --AND tgt.{target_current_flag_col} = true
         AND tgt.{target_eff_to_col} = CAST('9999-12-31 00:00:00' AS TIMESTAMP)
          WHEN MATCHED AND tgt.{target_attr_hash_col} <> src.new_hash THEN
            UPDATE SET
              --tgt.{target_current_flag_col} = false,
              tgt.{target_eff_to_col} = src.new_eff_from - INTERVAL 1 SECOND,
              tgt.{upd_by} = '{created_by}',
              tgt.{upd_ts} = current_timestamp()
        """

        log_message = f"creating expire_changed_current_query using ----- {expire_changed_current_query} "
        task_log(task_run_id=task_run_id,task_name=target_table,etl_component_name=etl_component_name,message=log_message)

        # logging.info("------ expire_changed_current_query ------")
        # logging.info(expire_changed_current_query)
        spark.sql(expire_changed_current_query)

        history_2 = spark.sql(f"DESCRIBE HISTORY {target_table}").first()
        metrics_2 = history_2.operationMetrics or {}

        update_dict = {"Inserted_2": metrics_2['numTargetRowsInserted'], "Updated_2": metrics_2['numTargetRowsUpdated'],"Deleted_2": metrics_2['numTargetRowsDeleted']}

        return update_dict

def insert_new_versions(
                            target_table: str,
                            src_view: str,
                            insert_cols: list,
                            match_cols: list,
                            batch_run_id,
                            task_run_id):
        """
        Inserts only if NOT MATCHED on (business_key + effective_from + attr_hash) for idempotency.
        """
        # <=> is used for handling nullable columns
        on_clause = " AND ".join([f"tgt.{c} <=> src.{c}" for c in match_cols])
        insert_col_list = ", ".join(insert_cols)
        values_list = ", ".join([f"src.{c}" for c in insert_cols])

        insert_new_versions_query = f"""
          MERGE INTO {target_table} AS tgt
          USING (
            SELECT
              *,
              '{created_by}' AS created_by,
              current_timestamp() AS created_ts,
              '{created_by}' AS last_updated_by,
              current_timestamp() AS last_updated_ts,
              true AS is_current
            FROM {src_view}
          ) AS src
          ON {on_clause}
          WHEN NOT MATCHED THEN
            INSERT ({insert_col_list})
            VALUES ({values_list})
        """
        log_message = f"creating insert_new_versions_query using ----- {insert_new_versions_query} "
        task_log(task_run_id=task_run_id,task_name=target_table,etl_component_name=etl_component_name,message=log_message)

        # logging.info("------ insert_new_versions_query ------")
        # logging.info(insert_new_versions_query)
        spark.sql(insert_new_versions_query)

        history_1 = spark.sql(f"DESCRIBE HISTORY {target_table}").first()
        metrics_1 = history_1.operationMetrics

        insert_dict = {"Inserted_1": metrics_1['numTargetRowsInserted'], "Updated_1": metrics_1['numTargetRowsUpdated'],"Deleted_1": metrics_1['numTargetRowsDeleted']}

        return insert_dict
    # ----------------------------------------------------
    # ### CHANGED: post-check supports composite keys
    # ----------------------------------------------------

def post_check_single_current( target_table: str, business_key_cols, current_flag_col: str):
        bk_cols = _as_list(business_key_cols)
        bk_sel = _key_select(bk_cols)

        post_check_single_current_query = f"""
          SELECT {bk_sel}, COUNT(*) AS current_cnt
          FROM {target_table}
          WHERE {current_flag_col} = true
          GROUP BY {bk_sel}
          HAVING COUNT(*) > 1
        """
        
        # logging.info("------ post_check_single_current_query ------")
        # logging.info(post_check_single_current_query)
        df = spark.sql(post_check_single_current_query)
        return df

def run_dimension( cfg: dict,batch_run_id,task_run_id):
        """
        cfg keys required:
          - target_table
          - stage_sql
          - business_key_col
          - watermark_col (in target)
          - stage_watermark_filter (optional: if stage_sql expects watermark parameter)
          - effective_from_col (in stage)
          - attribute_cols (list)
          - target_current_flag_col (default: is_current)
          - target_attr_hash_col (default: attr_hash)
          - target_eff_to_col (default: effective_to_ts)
          - insert_cols (list) if you want explicit control
        """
        etl_component_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        print(etl_component_name)

        try :
            target = cfg["target_table"]
            # logging.info(f"------ Running for {target} ------")

            # ### CHANGED: allow single or composite keys
            bk_cols = cfg.get("business_key_cols", cfg.get("business_key_col"))
            bk_cols = _as_list(bk_cols)

            # bk = cfg["business_key_col"]
            # wm_col = cfg.get("watermark_col", "dv_load_date")

            wm_col = cfg.get("watermark_col", "effective_from_ts")
            eff_from = cfg.get("effective_from_col", "effective_from_ts")

            target_current = cfg.get("target_current_flag_col", "is_current")
            target_hash = cfg.get("target_attr_hash_col", "attr_hash")
            target_eff_to = cfg.get("target_eff_to_col", "effective_to_ts")
            record_version_col = cfg.get("record_version_col", "record_version")

            audit_cols = {
                "last_updated_by": cfg.get("last_updated_by_col", "last_updated_by"),
                "last_updated_ts": cfg.get("last_updated_ts_col", "last_updated_ts"),
            }

            # 1) Watermark
            watermark = get_watermark(target, wm_col,batch_run_id,task_run_id)

            # 2) Stage SQL (inject watermark if placeholder exists)
            #stage_sql = cfg["stage_sql"].replace("${WATERMARK}", watermark)
            stage_sql = cfg["stage_sql"]
            # Replace watermark value placeholder
            stage_sql = stage_sql.replace(f"> ${{watermark_col}}", f"> '{watermark}'")
            stage_sql = stage_sql.replace(f"${{watermark_col}}", wm_col)
            stage_view = f"stg_{cfg['name']}_base"
            stage_hash_view = f"stg_{cfg['name']}_hash"
            stage_final_view = f"stg_{cfg['name']}_final"

            
            build_stage_view(stage_sql, stage_view,batch_run_id,task_run_id,target)
          
            from pyspark.sql import SparkSession
            spark = SparkSession.builder.getOrCreate()

            stg_count = spark.table(stage_view).limit(1).count()
            if stg_count == 0:
                is_stg_empty = 'Y'
            else:
                is_stg_empty = 'N'
            
            log_message = f"is stg_view empty : {is_stg_empty}"
            task_log(task_run_id=task_run_id,task_name=target,etl_component_name=etl_component_name,message=log_message)

            # skip if stage is empty
            if stg_count == 0:
              result_changes = {"Inserted_1": 0, "Updated_1": 0,"Deleted_1": 0}
              result_inserts = {"Inserted_2": 0, "Updated_2": 0,"Deleted_2": 0}
              result_dupes = {"watermark_used": watermark}
              task_status = {"task_status" : "success"}

              final_return_data = result_changes | result_inserts | result_dupes | task_status
              print ('-------- Final return data : ----------')
              print (final_return_data)
              log_message = f"stage_view is empty, therefore not running further process"
              task_log(task_run_id=task_run_id,task_name=target,etl_component_name=etl_component_name,message=log_message)

              return final_return_data

            # 3) Add attr_hash
            hash_cols = cfg["attribute_cols"]
            compute_attr_hash(stage_view, hash_cols, stage_hash_view,batch_run_id,task_run_id,target)

          # 4) record_version ()

            add_record_version(
                target_table=target,
                src_view=stage_hash_view,
                out_view=stage_final_view,
                business_key_cols=bk_cols,
                effective_from_col=eff_from,
                record_version_col=record_version_col,
                batch_run_id=batch_run_id,
                task_run_id=task_run_id
            )

            # 4) Expire changed currents
            result_changes = expire_changed_current(
                target_table=target,
                src_view=stage_final_view,
                business_key_cols=bk_cols,
                target_current_flag_col=target_current,
                target_attr_hash_col=target_hash,
                target_eff_to_col=target_eff_to,
                src_eff_from_col=eff_from,
                audit_cols=audit_cols,
                batch_run_id=batch_run_id,
                task_run_id=task_run_id
            )

            # print(result_changes)

            # 5) Insert new versions (idempotent)
            # match_cols define uniqueness of a version

            # match_cols = cfg.get("match_cols", [bk, eff_from, "attr_hash"])
            match_cols = cfg.get("match_cols", bk_cols + ["attr_hash"])

            # insert columns must match your dim schema; define per config for full control
            insert_cols = cfg["insert_cols"]

            result_inserts= insert_new_versions(
                target_table=target,
                #src_view=stage_hash_view,
                src_view=stage_final_view,
                insert_cols=insert_cols,
                match_cols=match_cols,
                batch_run_id = batch_run_id,
                task_run_id = task_run_id
            )
            
            # 6) Post checks
            # dupes_df = post_check_single_current(target, bk_cols, target_current)
            # result_dupes = {"watermark_used": watermark, "duplicate_current_df": dupes_df}
            result_dupes = {"watermark_used": watermark}
            task_status = {"task_status" : "success"}

            final_return_data = result_changes | result_inserts | result_dupes | task_status
            print(final_return_data)
            return final_return_data
        except Exception as e:
            notebook_name = etl_component_name
            task_status = {"task_status" : "failed"}
            # exception_message = e
            error_details = {"exception_message": str(e)}
            notebook_details = {"notebook_name":notebook_name}
            
            return_value = task_status | error_details | notebook_details
            print(return_value)
            return return_value

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
# ============================================================
# SCD TYPE 1 - Dimension Loader (Overwrite / Upsert)
# ============================================================

def merge_scd1(
    target_table: str,
    src_view: str,
    business_key_cols,
    attribute_cols: list,
    insert_cols: list,
    audit_cols: dict,
    batch_run_id,
    task_run_id
):
    """
    Single MERGE statement that:
      - UPDATES existing records when attr_hash differs (attributes changed)
      - INSERTS new records when business key not found in target
    """
    bk_cols = _as_list(business_key_cols)
    on_keys = _join_on_keys("tgt", "src", bk_cols)

    upd_by = audit_cols.get("last_updated_by", "last_updated_by")
    upd_ts = audit_cols.get("last_updated_ts", "last_updated_ts")

    # Build SET clause for all attribute columns and audit columns
    set_clause = ",\n              ".join([f"tgt.{c} = src.{c}" for c in attribute_cols])
    set_clause += f",\n              tgt.attr_hash = src.attr_hash"
    set_clause += f",\n              tgt.{upd_by} = '{created_by}'"
    set_clause += f",\n              tgt.{upd_ts} = current_timestamp()"

    insert_col_list = ", ".join(insert_cols)
    values_list = ", ".join([f"src.{c}" for c in insert_cols])

    print(src_view)
    # Build the MERGE SQL statement for SCD1 upsert
    merge_query = f"""
      MERGE INTO {target_table} AS tgt
      USING (
        SELECT
          *,
          '{created_by}' AS created_by,
          --current_timestamp() AS created_ts,
          '{created_by}' AS last_updated_by,
          current_timestamp() AS last_updated_ts
        FROM {src_view}
      ) AS src
      ON {on_keys}
      WHEN MATCHED AND tgt.attr_hash <> src.attr_hash THEN
        UPDATE SET
              {set_clause}
      WHEN NOT MATCHED THEN
        INSERT ({insert_col_list})
        VALUES ({values_list})
    """

    # Log the merge query for audit/debugging
    log_message = f"creating merge_scd1 query ----- {merge_query}"
    task_log(task_run_id=task_run_id, task_name=target_table, etl_component_name=etl_component_name, message=log_message)
    spark.sql(merge_query)

    # Get operation metrics from Delta table history
    history = spark.sql(f"DESCRIBE HISTORY {target_table}").first()
    metrics = history.operationMetrics or {}

    return {
        "Inserted": metrics.get('numTargetRowsInserted', 0),
        "Updated": metrics.get('numTargetRowsUpdated', 0),
        "Deleted": metrics.get('numTargetRowsDeleted', 0),
    }


def run_dimension_scd1(cfg: dict, batch_run_id, task_run_id):
    """
    SCD1 orchestrator. cfg keys required:
      - target_table
      - name (short name for temp view naming)
      - stage_sql
      - business_key_col (single) or business_key_cols (list)
      - watermark_col
      - attribute_cols (list of columns to track for changes)
      - insert_cols (list of columns for INSERT)
    """
    # Get notebook path for logging
    etl_component_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(etl_component_name)

    try:
        target = cfg["target_table"]

        # Get business key columns (single or composite)
        bk_cols = cfg.get("business_key_cols", cfg.get("business_key_col"))
        bk_cols = _as_list(bk_cols)

        wm_col = cfg.get("watermark_col", "load_date")

        # Define audit columns for tracking updates
        audit_cols = {
            "last_updated_by": cfg.get("last_updated_by_col", "last_updated_by"),
            "last_updated_ts": cfg.get("last_updated_ts_col", "last_updated_ts"),
        }

        # 1) Get watermark for incremental load
        watermark = get_watermark(target, wm_col, batch_run_id, task_run_id)

        # 2) Build stage view from SQL, inject watermark
        stage_sql = cfg["stage_sql"]
        stage_sql = stage_sql.replace(f"> ${{watermark_col}}", f"> '{watermark}'")
        stage_sql = stage_sql.replace(f"${{watermark_col}}", wm_col)

        stage_view = f"stg_{cfg['name']}_base"
        stage_hash_view = f"stg_{cfg['name']}_hash"

        build_stage_view(stage_sql, stage_view, batch_run_id, task_run_id, target)

        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()

        # Check if stage view is empty
        stg_count = spark.table(stage_view).limit(1).count()

        log_message = f"is stg_view empty : {'Y' if stg_count == 0 else 'N'}"
        task_log(task_run_id=task_run_id, task_name=target, etl_component_name=etl_component_name, message=log_message)

        # Skip further processing if stage is empty
        if stg_count == 0:
            final_return_data = {
                "Inserted": 0, "Updated": 0, "Deleted": 0,
                "watermark_used": watermark,
                "task_status": "success"
            }
            print('-------- Final return data : ----------')
            print(final_return_data)
            log_message = "stage_view is empty, therefore not running further process"
            task_log(task_run_id=task_run_id, task_name=target, etl_component_name=etl_component_name, message=log_message)
            return final_return_data

        # 3) Compute attr_hash for change detection
        hash_cols = cfg["attribute_cols"]
        compute_attr_hash(stage_view, hash_cols, stage_hash_view, batch_run_id, task_run_id, target)

        # 4) MERGE: Update changed + Insert new (single statement)
        insert_cols = cfg["insert_cols"]

        result_merge = merge_scd1(
            target_table=target,
            src_view=stage_hash_view,
            business_key_cols=bk_cols,
            attribute_cols=hash_cols,
            insert_cols=insert_cols,
            audit_cols=audit_cols,
            batch_run_id=batch_run_id,
            task_run_id=task_run_id
        )

        # Combine results and return
        final_return_data = result_merge | {"watermark_used": watermark, "task_status": "success"}
        print(final_return_data)
        return final_return_data

    except Exception as e:
        notebook_name = etl_component_name
        return_value = {
            "task_status": "failed",
            "exception_message": str(e),
            "notebook_name": notebook_name
        }
        print(return_value)
        return return_value

In [0]:
# ============================================================
# BUSINESS VAULT - SCD2-style Loader
# ============================================================
# This loader expects table-specific business logic to be handled
# in cfg['stage_sql']. It only handles watermarking, staging,
# hash-based change detection, closing old current rows, and
# inserting new current rows.

def _bv_metric_dict(target_table: str, prefix: str = ""):
    history = spark.sql(f"DESCRIBE HISTORY {target_table}").first()
    metrics = history.operationMetrics or {}
    return {
        f"{prefix}Inserted": metrics.get("numTargetRowsInserted", 0),
        f"{prefix}Updated": metrics.get("numTargetRowsUpdated", 0),
        f"{prefix}Deleted": metrics.get("numTargetRowsDeleted", 0),
    }


def _hash_expr_for_alias(alias: str, hash_cols: list):
    return "sha2(concat_ws('||'," + ",".join([f"coalesce(cast({alias}.{c} as string),'')" for c in hash_cols]) + "),256)"


def expire_bv_changed_current(
    target_table: str,
    src_view: str,
    business_key_cols,
    target_attr_hash_col: str,
    target_eff_to_col: str,
    target_current_flag_col: str,
    src_eff_from_col: str,
    audit_cols: dict,
    batch_run_id,
    task_run_id,
    open_end_ts: str = "9999-12-31 00:00:00",
    attribute_cols=None,
    persist_attr_hash: bool = True,
    src_eff_from_expr: str = None,
    current_true_value: str = "true",
    current_false_value: str = "false",
):
    bk_cols = _as_list(business_key_cols)
    bk_sel = _key_select(bk_cols)
    on_keys = _join_on_keys("tgt", "src", bk_cols)
    attribute_cols = attribute_cols or []

    effective_from_source = src_eff_from_expr or src_eff_from_col
    src_select_cols = [bk_sel, f"{effective_from_source} AS new_eff_from"]
    if persist_attr_hash:
        src_select_cols.append("attr_hash AS new_hash")
        change_condition = f"COALESCE(tgt.{target_attr_hash_col}, '') <> COALESCE(src.new_hash, '')"
    else:
        src_select_cols.append("attr_hash AS new_hash")
        tgt_hash_expr = _hash_expr_for_alias("tgt", attribute_cols) if attribute_cols else "''"
        change_condition = f"COALESCE({tgt_hash_expr}, '') <> COALESCE(src.new_hash, '')"

    update_assignments = [
        f"tgt.{target_eff_to_col} = src.new_eff_from - INTERVAL 1 SECOND",
        f"tgt.{target_current_flag_col} = {current_false_value}",
    ]
    upd_by = audit_cols.get("last_updated_by")
    upd_ts = audit_cols.get("last_updated_ts")
    if upd_by:
        update_assignments.append(f"tgt.{upd_by} = '{created_by}'")
    if upd_ts:
        update_assignments.append(f"tgt.{upd_ts} = current_timestamp()")

    expire_query = f"""
      MERGE INTO {target_table} AS tgt
      USING (
        SELECT {", ".join(src_select_cols)}
        FROM {src_view}
      ) AS src
      ON {on_keys}
     AND tgt.{target_eff_to_col} = CAST('{open_end_ts}' AS TIMESTAMP)
     AND tgt.{target_current_flag_col} = {current_true_value}
      WHEN MATCHED AND ({change_condition}) THEN
        UPDATE SET
          {", ".join(update_assignments)}
    """

    task_log(
        task_run_id=task_run_id,
        task_name=target_table,
        etl_component_name=etl_component_name,
        message=f"creating expire_bv_changed_current query ----- {expire_query}",
    )
    spark.sql(expire_query)
    return _bv_metric_dict(target_table, "Expired_")



def insert_bv_new_current(
    target_table: str,
    src_view: str,
    insert_cols: list,
    match_cols: list,
    effective_from_col: str,
    effective_to_col: str,
    current_flag_col: str,
    batch_run_id,
    task_run_id,
    open_end_ts: str = "9999-12-31 00:00:00",
    generate_effective_from: bool = False,
    effective_from_expr: str = "current_timestamp()",
    current_true_value: str = "true",
    business_key_cols=None,
    attribute_cols=None,
    persist_attr_hash: bool = True,
    target_attr_hash_col: str = "attr_hash",
):
    if business_key_cols:
        on_parts = [f"tgt.{c} <=> src.{c}" for c in _as_list(business_key_cols)]
        on_parts.append(f"tgt.{current_flag_col} = {current_true_value}")
        if persist_attr_hash:
            on_parts.append(f"COALESCE(tgt.{target_attr_hash_col}, '') <=> COALESCE(src.attr_hash, '')")
        else:
            tgt_hash_expr = _hash_expr_for_alias("tgt", attribute_cols or []) if attribute_cols else "''"
            on_parts.append(f"COALESCE({tgt_hash_expr}, '') <=> COALESCE(src.attr_hash, '')")
    else:
        on_parts = []
        for col in match_cols:
            if generate_effective_from and col.lower() == effective_from_col.lower():
                on_parts.append(f"tgt.{col} <=> src.__bv_effective_from")
            else:
                on_parts.append(f"tgt.{col} <=> src.{col}")
    on_clause = " AND ".join(on_parts)
    insert_col_list = ", ".join(insert_cols)

    values = []
    for col in insert_cols:
        col_key = col.lower()
        if col_key == effective_from_col.lower() and generate_effective_from:
            values.append("src.__bv_effective_from")
        elif col_key == effective_to_col.lower():
            values.append("src.__bv_effective_to")
        elif col_key == current_flag_col.lower():
            values.append("src.__bv_current_flag")
        elif col_key == "attr_hash":
            values.append("src.attr_hash")
        elif col_key == "created_by":
            values.append("src.__bv_created_by")
        elif col_key == "created_ts":
            values.append("src.__bv_created_ts")
        elif col_key == "last_updated_by":
            values.append("src.__bv_last_updated_by")
        elif col_key == "last_updated_ts":
            values.append("src.__bv_last_updated_ts")
        else:
            values.append(f"src.{col}")
    values_list = ", ".join(values)

    effective_from_select = f",\n          {effective_from_expr} AS __bv_effective_from" if generate_effective_from else ""
    insert_query = f"""
      MERGE INTO {target_table} AS tgt
      USING (
        SELECT
          *,
          CAST('{open_end_ts}' AS TIMESTAMP) AS __bv_effective_to,
          {current_true_value} AS __bv_current_flag,
          '{created_by}' AS __bv_created_by,
          current_timestamp() AS __bv_created_ts,
          '{created_by}' AS __bv_last_updated_by,
          current_timestamp() AS __bv_last_updated_ts{effective_from_select}
        FROM {src_view}
      ) AS src
      ON {on_clause}
      WHEN NOT MATCHED THEN
        INSERT ({insert_col_list})
        VALUES ({values_list})
    """

    task_log(
        task_run_id=task_run_id,
        task_name=target_table,
        etl_component_name=etl_component_name,
        message=f"creating insert_bv_new_current query ----- {insert_query}",
    )
    spark.sql(insert_query)
    return _bv_metric_dict(target_table, "New_")



def update_bv_watermark(target_table: str, src_view: str, watermark_col: str, batch_run_id, task_run_id):
    update_query = f"""
      MERGE INTO {full_wm_table_name} AS tgt
      USING (
        SELECT '{target_table}' AS table_name, MAX({watermark_col}) AS watermark
        FROM {src_view}
      ) AS src
      ON tgt.table_name = src.table_name
      WHEN MATCHED AND src.watermark IS NOT NULL THEN
        UPDATE SET tgt.watermark = src.watermark
      WHEN NOT MATCHED AND src.watermark IS NOT NULL THEN
        INSERT (table_name, watermark) VALUES (src.table_name, src.watermark)
    """
    task_log(
        task_run_id=task_run_id,
        task_name=target_table,
        etl_component_name=etl_component_name,
        message=f"creating update_bv_watermark query ----- {update_query}",
    )
    spark.sql(update_query)


def run_business_vault_scd2(cfg: dict, batch_run_id, task_run_id):
    """
    Generic Business Vault SCD2-style loader.

    Required cfg keys:
      - name
      - target_table
      - stage_sql
      - business_key_col or business_key_cols
      - attribute_cols
      - insert_cols

    Optional cfg keys:
      - watermark_col, default load_date
      - effective_from_col, default effective_from
      - effective_to_col, default effective_to
      - current_flag_col, default current_active_flag
      - persist_attr_hash, default False
      - target_attr_hash_col, default attr_hash
      - generate_effective_from, default True
      - effective_from_expr, default current_date()
      - match_cols, default business keys + effective_from + attr_hash when persisted,
        otherwise business keys + effective_from + attribute_cols
      - open_end_ts, default 9999-12-31 00:00:00
      - current_true_value, default 'Y'
      - current_false_value, default 'N'
      - update_watermark, default True when a watermark/effective column is available
    """
    global etl_component_name
    etl_component_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

    try:
        target = cfg["target_table"]
        bk_cols = _as_list(cfg.get("business_key_cols", cfg.get("business_key_col")))
        wm_col = cfg.get("watermark_col", "load_date")
        eff_from = cfg.get("effective_from_col", "effective_from")
        eff_to = cfg.get("effective_to_col", "effective_to")
        current_flag = cfg.get("current_flag_col", "current_active_flag")
        target_hash = cfg.get("target_attr_hash_col", "attr_hash")
        open_end_ts = cfg.get("open_end_ts", "9999-12-31 00:00:00")
        persist_attr_hash = cfg.get("persist_attr_hash", False)
        generate_effective_from = cfg.get("generate_effective_from", True)
        effective_from_expr = cfg.get("effective_from_expr", "current_date()")
        current_true_value = cfg.get("current_true_value", "'Y'")
        current_false_value = cfg.get("current_false_value", "'N'")

        audit_cols = {}
        if cfg.get("last_updated_by_col"):
            audit_cols["last_updated_by"] = cfg.get("last_updated_by_col")
        if cfg.get("last_updated_ts_col"):
            audit_cols["last_updated_ts"] = cfg.get("last_updated_ts_col")

        watermark = get_watermark(target, wm_col, batch_run_id, task_run_id)
        stage_sql = cfg["stage_sql"]
        stage_sql = stage_sql.replace(f"> ${{watermark_col}}", f"> '{watermark}'")
        stage_sql = stage_sql.replace(f"${{watermark_col}}", wm_col)

        stage_view = f"stg_bv_{cfg['name']}_base"
        stage_effective_view = f"stg_bv_{cfg['name']}_effective"
        stage_hash_view = f"stg_bv_{cfg['name']}_hash"

        build_stage_view(stage_sql, stage_view, batch_run_id, task_run_id, target)

        from pyspark.sql import SparkSession
        spark_session = SparkSession.builder.getOrCreate()
        stg_count = spark_session.table(stage_view).limit(1).count()

        task_log(
            task_run_id=task_run_id,
            task_name=target,
            etl_component_name=etl_component_name,
            message=f"is business vault stage_view empty : {'Y' if stg_count == 0 else 'N'}",
        )

        if stg_count == 0:
            final_return_data = {
                "Expired_Inserted": 0,
                "Expired_Updated": 0,
                "Expired_Deleted": 0,
                "New_Inserted": 0,
                "New_Updated": 0,
                "New_Deleted": 0,
                "watermark_used": watermark,
                "task_status": "success",
            }
            print(final_return_data)
            return final_return_data

        hash_input_view = stage_view
        if generate_effective_from:
            control_cols = {eff_from.lower(), eff_to.lower(), current_flag.lower()}
            stage_cols = [c for c in spark_session.table(stage_view).columns if c.lower() not in control_cols]
            stage_select_cols = ", ".join([f"`{c}`" for c in stage_cols])
            effective_query = f"""
              CREATE OR REPLACE TEMP VIEW {stage_effective_view} AS
              SELECT {stage_select_cols}, {effective_from_expr} AS {eff_from}
              FROM {stage_view}
            """
            task_log(
                task_run_id=task_run_id,
                task_name=target,
                etl_component_name=etl_component_name,
                message=f"creating business vault effective_from view ----- {effective_query}",
            )
            spark.sql(effective_query)
            hash_input_view = stage_effective_view

        compute_attr_hash(hash_input_view, cfg["attribute_cols"], stage_hash_view, batch_run_id, task_run_id, target)

        result_expired = expire_bv_changed_current(
            target_table=target,
            src_view=stage_hash_view,
            business_key_cols=bk_cols,
            target_attr_hash_col=target_hash,
            target_eff_to_col=eff_to,
            target_current_flag_col=current_flag,
            src_eff_from_col=eff_from,
            audit_cols=audit_cols,
            batch_run_id=batch_run_id,
            task_run_id=task_run_id,
            open_end_ts=open_end_ts,
            attribute_cols=cfg["attribute_cols"],
            persist_attr_hash=persist_attr_hash,
            current_true_value=current_true_value,
            current_false_value=current_false_value,
        )

        if persist_attr_hash:
            default_match_cols = bk_cols + [eff_from, "attr_hash"]
        else:
            default_match_cols = bk_cols + [eff_from] + cfg["attribute_cols"]
        match_cols = cfg.get("match_cols", default_match_cols)

        result_new = insert_bv_new_current(
            target_table=target,
            src_view=stage_hash_view,
            insert_cols=cfg["insert_cols"],
            match_cols=match_cols,
            effective_from_col=eff_from,
            effective_to_col=eff_to,
            current_flag_col=current_flag,
            batch_run_id=batch_run_id,
            task_run_id=task_run_id,
            open_end_ts=open_end_ts,
            generate_effective_from=False,
            effective_from_expr=effective_from_expr,
            current_true_value=current_true_value,
            business_key_cols=bk_cols,
            attribute_cols=cfg["attribute_cols"],
            persist_attr_hash=persist_attr_hash,
            target_attr_hash_col=target_hash,
        )

        if cfg.get("update_watermark", True):
            stage_hash_cols = [c.lower() for c in spark_session.table(stage_hash_view).columns]
            watermark_update_col = None
            if wm_col.lower() in stage_hash_cols:
                watermark_update_col = wm_col
            elif eff_from.lower() in stage_hash_cols:
                watermark_update_col = eff_from

            if watermark_update_col:
                update_bv_watermark(target, stage_hash_view, watermark_update_col, batch_run_id, task_run_id)
            else:
                task_log(
                    task_run_id=task_run_id,
                    task_name=target,
                    etl_component_name=etl_component_name,
                    message=f"skipping watermark update because neither {wm_col} nor {eff_from} exists in {stage_hash_view}",
                )

        final_return_data = result_expired | result_new | {"watermark_used": watermark, "task_status": "success"}
        print(final_return_data)
        return final_return_data

    except Exception as e:
        return_value = {
            "task_status": "failed",
            "exception_message": str(e),
            "notebook_name": etl_component_name,
        }
        print(return_value)
        return return_value



def insert_bv_xref_rows(
    target_table: str,
    src_view: str,
    insert_cols: list,
    match_cols: list,
    batch_run_id,
    task_run_id,
):
    """
    Insert-only XREF load. Existing mappings are not updated or expired.
    Use match_cols to define mapping uniqueness, for example:
      - SOURCE_SYSTEM + SOURCE_PERSON_ID
      - SOURCE_SYSTEM + SOURCE_PERSON_ID + MASTER_PERSON_HK
    """
    on_clause = " AND ".join([f"tgt.{c} <=> src.{c}" for c in match_cols])
    insert_col_list = ", ".join(insert_cols)

    values = []
    for col in insert_cols:
        if col == "created_by":
            values.append("src.__xref_created_by")
        elif col == "created_ts":
            values.append("src.__xref_created_ts")
        elif col == "last_updated_by":
            values.append("src.__xref_last_updated_by")
        elif col == "last_updated_ts":
            values.append("src.__xref_last_updated_ts")
        else:
            values.append(f"src.{col}")
    values_list = ", ".join(values)

    insert_query = f"""
      MERGE INTO {target_table} AS tgt
      USING (
        SELECT
          *,
          '{created_by}' AS __xref_created_by,
          current_timestamp() AS __xref_created_ts,
          '{created_by}' AS __xref_last_updated_by,
          current_timestamp() AS __xref_last_updated_ts
        FROM {src_view}
      ) AS src
      ON {on_clause}
      WHEN NOT MATCHED THEN
        INSERT ({insert_col_list})
        VALUES ({values_list})
    """

    task_log(
        task_run_id=task_run_id,
        task_name=target_table,
        etl_component_name=etl_component_name,
        message=f"creating insert_bv_xref_rows query ----- {insert_query}",
    )
    spark.sql(insert_query)
    return _bv_metric_dict(target_table, "Xref_")


def run_business_vault_xref(cfg: dict, batch_run_id, task_run_id):
    """
    Generic insert-only Business Vault XREF loader.

    Required cfg keys:
      - name
      - target_table
      - stage_sql
      - match_cols
      - insert_cols

    Optional cfg keys:
      - watermark_col, default load_ts
      - update_watermark, default False
      - conflict_check_cols, default []

    This function intentionally has no WHEN MATCHED UPDATE and no SCD2 expiry.
    """
    global etl_component_name
    etl_component_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

    try:
        target = cfg["target_table"]
        match_cols = cfg["match_cols"]
        wm_col = cfg.get("watermark_col", "load_ts")

        watermark = get_watermark(target, wm_col, batch_run_id, task_run_id)
        stage_sql = cfg["stage_sql"]
        stage_sql = stage_sql.replace(f"> ${{watermark_col}}", f"> '{watermark}'")
        stage_sql = stage_sql.replace(f"${{watermark_col}}", wm_col)

        stage_view = f"stg_bv_xref_{cfg['name']}_base"
        build_stage_view(stage_sql, stage_view, batch_run_id, task_run_id, target)

        from pyspark.sql import SparkSession
        spark_session = SparkSession.builder.getOrCreate()
        stg_count = spark_session.table(stage_view).limit(1).count()

        task_log(
            task_run_id=task_run_id,
            task_name=target,
            etl_component_name=etl_component_name,
            message=f"is business vault xref stage_view empty : {'Y' if stg_count == 0 else 'N'}",
        )

        if stg_count == 0:
            final_return_data = {
                "Xref_Inserted": 0,
                "Xref_Updated": 0,
                "Xref_Deleted": 0,
                "watermark_used": watermark,
                "task_status": "success",
            }
            print(final_return_data)
            return final_return_data

        conflict_cols = cfg.get("conflict_check_cols", [])
        if conflict_cols:
            key_cols = ", ".join(match_cols)
            conflict_expr = " OR ".join([f"NOT (tgt.{c} <=> src.{c})" for c in conflict_cols])
            conflict_query = f"""
              SELECT src.*
              FROM {stage_view} src
              INNER JOIN {target} tgt
                ON {" AND ".join([f"tgt.{c} <=> src.{c}" for c in match_cols])}
              WHERE {conflict_expr}
              LIMIT 10
            """
            conflict_count = spark.sql(conflict_query).count()
            if conflict_count > 0:
                raise Exception(f"XREF conflict detected for {target}; same match key has different mapped values")

        result_insert = insert_bv_xref_rows(
            target_table=target,
            src_view=stage_view,
            insert_cols=cfg["insert_cols"],
            match_cols=match_cols,
            batch_run_id=batch_run_id,
            task_run_id=task_run_id,
        )

        if cfg.get("update_watermark", False):
            update_bv_watermark(target, stage_view, wm_col, batch_run_id, task_run_id)

        final_return_data = result_insert | {"watermark_used": watermark, "task_status": "success"}
        print(final_return_data)
        return final_return_data

    except Exception as e:
        return_value = {
            "task_status": "failed",
            "exception_message": str(e),
            "notebook_name": etl_component_name,
        }
        print(return_value)
        return return_value
